# Simulation 2 — robustness (three branches: gllvm / log1p / Huber)

ZQE picks the statistic `T(y)`, and the centering term `m₂ = E_θ[T·η]` supplies the
Fisher-consistency correction for *any* `T` automatically — so we can make the
estimator robust **by design** without losing consistency. Three branches, same
misspecified Poisson model, same contaminated data, differing only in influence:

| branch | `T(y)` | influence in `y` |
|---|---|---|
| `gllvm` | full Poisson likelihood | **unbounded** |
| `zqe` | `log1p(y)` | logarithmic |
| `zqe_huber` | `min(log1p(y), c)` | **bounded** (Huberised; `c = median+3·MAD` of `log1p y`) |

**Goal:** see the bounded arm **flat in outlier magnitude `M`** (bounded
influence) and **flat in fraction `eps` until its breakdown point** (when the
robust scale `c` itself corrupts, `eps ≈ 0.5`). See [`description.md`](description.md).

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
HERE = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
if not os.path.exists(os.path.join(HERE, "sweep.py")):
    HERE = os.path.join(os.getcwd(), "simulations", "simulation_2")
sys.path.insert(0, HERE)
import torch, sweep
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device     :", DEV)
print("clean model: q=%d p=%d n=%d" % (sweep.Q, sweep.P, sweep.N))
print("eps line   :", sweep.EPS_GRID, "at M =", sweep.M_FIX)
print("mag line   :", sweep.M_GRID, "at eps =", sweep.EPS_FIX)
print("conditions :", len(sweep.CONDITIONS))

## Demo: one contaminated rep (eps=0.05, M=1000)

In [ ]:
from gllvm.r_gllvm import RGllvm
r = RGllvm(method="VA", family="poisson", timeout=60)
assert r.available(), f"R not found at {r.rscript!r}"
demo = sweep.run_condition(eps=0.05, M=1000, rep=0, device=DEV, rgllvm=r)
demo.drop_duplicates("method")[["method", "eps", "M", "failed", "time_sec", "procrustes"]]

## Run the sweep (resumable; gllvm may DNF under heavy contamination)

In [ ]:
sweep.run_sweep(reps=20, device=DEV)

## Overview

In [ ]:
import pandas as pd
df = sweep.load_results()
fits = (df[df.method != "true"].drop_duplicates(["eps", "M", "rep", "method"])
        [["eps", "M", "rep", "method", "failed", "time_sec", "procrustes"]])
print("failures:", int(fits.failed.sum()), "/", len(fits))
(fits.groupby(["eps", "M", "method"])
 .agg(procrustes=("procrustes", "median"), fails=("failed", "sum")).round(3))